In [ ]:
# ============================================================
# CELL 1 — LOW RAM / GPU CHECK
# ============================================================

import sys
import subprocess

print("=" * 70)
print("HUNYUAN3D-2MV COLAB")
print("=" * 70)
print("System Python:", sys.version)

print("\nGPU:")
subprocess.run(["nvidia-smi"], check=False)

print("\nRAM:")
subprocess.run(["free", "-h"], check=False)

print("\nSWAP:")
subprocess.run(["swapon", "--show"], check=False)


In [ ]:
# ============================================================
# CELL 2 — CREATE PYTHON 3.10 VENV
# ============================================================

from pathlib import Path
import shutil
import subprocess

ENV = Path("/content/hunyuan_mv_env")
PYTHON = ENV / "bin" / "python"

if ENV.exists():
    print("Removing old environment...")
    shutil.rmtree(ENV, ignore_errors=True)

print("Creating Python 3.10 venv...")
subprocess.run(["python3.10", "-m", "venv", str(ENV)], check=True)

subprocess.run([
    str(PYTHON), "-m", "pip", "install", "-U",
    "pip", "setuptools", "wheel"
], check=True)

print(subprocess.check_output([str(PYTHON), "--version"], text=True).strip())
print("PYTHON:", PYTHON)


In [ ]:
# ============================================================
# CELL 3 — CLONE OFFICIAL HUNYUAN3D-2
# ============================================================

from pathlib import Path
import shutil
import subprocess

REPO = Path("/content/Hunyuan3D-2")

if REPO.exists():
    print("Removing old repository...")
    shutil.rmtree(REPO, ignore_errors=True)

subprocess.run([
    "git", "clone", "--recurse-submodules",
    "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git",
    str(REPO)
], check=True)

subprocess.run([
    "git", "submodule", "update",
    "--init", "--recursive"
], cwd=str(REPO), check=True)

print("REPO:", REPO)


In [ ]:
# ============================================================
# CELL 4 — INSTALL PYTORCH + HY3DGEN DEPENDENCIES
# ============================================================

import subprocess

PYTHON = "/content/hunyuan_mv_env/bin/python"
REPO = "/content/Hunyuan3D-2"

subprocess.run([
    PYTHON, "-m", "pip", "install",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "--index-url",
    "https://download.pytorch.org/whl/cu124"
], check=True)

subprocess.run([
    PYTHON, "-m", "pip", "install",
    "-e", REPO, "--no-deps"
], cwd=REPO, check=True)

packages = [
    "diffusers", "transformers", "accelerate", "einops", "omegaconf",
    "opencv-python", "numpy", "pillow", "trimesh", "pymeshlab",
    "pygltflib", "xatlas", "timm", "pyyaml", "ninja", "pybind11",
    "rembg", "onnxruntime", "fastapi", "uvicorn", "python-multipart",
    "psutil", "huggingface_hub", "safetensors", "pyngrok", "requests"
]

subprocess.run([
    PYTHON, "-m", "pip", "install", "-U", *packages
], check=True)

print("INSTALL COMPLETE")


In [ ]:
# ============================================================
# CELL 5 — VERIFY ENVIRONMENT
# ============================================================

import subprocess

PYTHON = "/content/hunyuan_mv_env/bin/python"
REPO = "/content/Hunyuan3D-2"

TEST = r'''
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

import hy3dgen
print("hy3dgen: OK")

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print("Hunyuan pipeline: OK")

import trimesh
print("trimesh: OK")

import rembg
print("rembg: OK")

import fastapi
print("fastapi: OK")

import pyngrok
print("pyngrok: OK")

print("ALL CHECKS PASSED")
'''

subprocess.run([PYTHON, "-c", TEST], cwd=REPO, check=True)


In [ ]:
# ============================================================
# CELL 6 — CREATE MULTI-VIEW API SERVER
# ============================================================

from pathlib import Path
import subprocess

PYTHON = "/content/hunyuan_mv_env/bin/python"
REPO = "/content/Hunyuan3D-2"
BASE = Path("/content/hunyuan3d_mv_server")
BASE.mkdir(parents=True, exist_ok=True)

SCRIPT = BASE / "server.py"
LOG = BASE / "server.log"

# Stop only our previous server.
subprocess.run(
    ["pkill", "-f", str(SCRIPT)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False
)

SERVER_CODE = r'''\
import os
import sys
import gc
import uuid
import traceback
from pathlib import Path
from typing import Optional

import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
import uvicorn

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

BASE = Path("/content/hunyuan3d_mv_server")
UPLOADS = BASE / "uploads"
OUTPUTS = BASE / "outputs"
UPLOADS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print("=" * 70, flush=True)
print("HUNYUAN3D-2MV TURBO SERVER", flush=True)
print("=" * 70, flush=True)
print("Python:", sys.version, flush=True)
print("Torch:", torch.__version__, flush=True)
print("CUDA:", torch.cuda.is_available(), flush=True)
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB", flush=True)

print("Loading tencent/Hunyuan3D-2mv ...", flush=True)

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2mv",
    subfolder="hunyuan3d-dit-v2-mv-turbo",
    variant="fp16",
    use_safetensors=True,
    device="cuda",
)

print("MODEL LOAD OK", flush=True)
print("VRAM allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB", flush=True)

try:
    pipeline.enable_flashvdm()
    print("FlashVDM: ENABLED", flush=True)
except Exception as e:
    print("FlashVDM: NOT ENABLED:", repr(e), flush=True)

app = FastAPI(title="Hunyuan3D-2mv Multi-View API")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "tencent/Hunyuan3D-2mv",
        "mode": "multi-view",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0),
        "vram_allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 2),
    }

@app.post("/generate_multi")
async def generate_multi(
    front: Optional[UploadFile] = File(None),
    left: Optional[UploadFile] = File(None),
    back: Optional[UploadFile] = File(None),
    right: Optional[UploadFile] = File(None),
):
    incoming = {
        "front": front,
        "left": left,
        "back": back,
        "right": right,
    }

    views = {}
    temp_files = []

    try:
        for role, upload in incoming.items():
            if upload is None:
                continue

            filename = Path(upload.filename or f"{role}.png").name
            path = UPLOADS / f"{uuid.uuid4().hex[:8]}_{filename}"
            path.write_bytes(await upload.read())
            temp_files.append(path)

            try:
                views[role] = Image.open(path).convert("RGBA")
            except Exception as e:
                raise HTTPException(status_code=400, detail=f"Invalid image for {role}: {e}")

        if len(views) < 2:
            raise HTTPException(
                status_code=400,
                detail="Provide at least 2 views: front/back/left/right",
            )

        print("=" * 70, flush=True)
        print("MULTI-VIEW GENERATION", flush=True)
        print("Views:", list(views.keys()), flush=True)
        print("=" * 70, flush=True)

        with torch.inference_mode():
            mesh = pipeline(
                image=views,
                num_inference_steps=5,
                octree_resolution=380,
                num_chunks=20000,
                generator=torch.manual_seed(12345),
                output_type="trimesh",
            )[0]

        print("MESH GENERATED", flush=True)

        filename = "multiview_" + uuid.uuid4().hex[:10] + ".glb"
        output = OUTPUTS / filename
        mesh.export(str(output))

        print("GLB CREATED:", output, flush=True)

        del mesh
        for image in views.values():
            try:
                image.close()
            except Exception:
                pass
        del views
        gc.collect()
        torch.cuda.empty_cache()

        return {
            "status": "completed",
            "filename": filename,
            "result": "/result/" + filename,
        }

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        for path in temp_files:
            try:
                path.unlink()
            except Exception:
                pass

@app.get("/result/{filename}")
def result(filename: str):
    safe = Path(filename).name
    path = OUTPUTS / safe
    if not path.exists():
        raise HTTPException(status_code=404, detail="Result not found")
    return FileResponse(str(path), media_type="model/gltf-binary", filename=safe)

print("Starting API on 127.0.0.1:8001", flush=True)
uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")
'''

SCRIPT.write_text(SERVER_CODE)

subprocess.run(
    [PYTHON, "-m", "py_compile", str(SCRIPT)],
    check=True
)

print("SYNTAX: OK")
print("SCRIPT:", SCRIPT)
print("LOG:", LOG)


In [ ]:
# ============================================================
# CELL 7 — START SERVER + WAIT FOR HEALTH
# ============================================================

import subprocess
import time
from pathlib import Path
import requests

PYTHON = "/content/hunyuan_mv_env/bin/python"
SCRIPT = "/content/hunyuan3d_mv_server/server.py"
LOG = Path("/content/hunyuan3d_mv_server/server.log")

log_handle = open(LOG, "w", buffering=1)

process = subprocess.Popen(
    [PYTHON, "-u", SCRIPT],
    cwd="/content/Hunyuan3D-2",
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Started PID:", process.pid)
print("LOG:", LOG)

start = time.time()
timeout = 1200

while True:

    if process.poll() is not None:
        log_handle.flush()
        log_handle.close()
        print(LOG.read_text(errors="replace")[-30000:] if LOG.exists() else "No log")
        raise RuntimeError(
            f"Hunyuan3D-2mv process exited: {process.returncode}"
        )

    try:
        r = requests.get(
            "http://127.0.0.1:8001/health",
            timeout=5,
        )
        if r.status_code == 200:
            print("=" * 70)
            print("✅ HUNYUAN3D-2MV API READY")
            print("=" * 70)
            print(r.json())
            break
    except Exception:
        pass

    if time.time() - start > timeout:
        log_handle.flush()
        log_handle.close()
        print(LOG.read_text(errors="replace")[-30000:] if LOG.exists() else "No log")
        raise TimeoutError("Hunyuan3D-2mv API startup timeout")

    time.sleep(5)


In [ ]:
# ============================================================
# CELL 8 — NGROK PUBLIC URL
# ============================================================

import requests
from pyngrok import ngrok

NGROK_TOKEN = "ใส่_NGROK_TOKEN_ของคุณ"

if not NGROK_TOKEN or NGROK_TOKEN.startswith("ใส่_"):
    raise RuntimeError("ใส่ NGROK_TOKEN ใน Cell 8 ก่อน Run All")

local = requests.get(
    "http://127.0.0.1:8001/health",
    timeout=20,
)
local.raise_for_status()
print("LOCAL:", local.json())

ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(
    addr=8001,
    proto="http",
)

PUBLIC_URL = str(tunnel.public_url)

if PUBLIC_URL.startswith("http://"):
    PUBLIC_URL = "https://" + PUBLIC_URL[7:]

print("=" * 70)
print("✅ NGROK READY")
print("=" * 70)
print("PUBLIC URL:", PUBLIC_URL)
print("HEALTH:", PUBLIC_URL + "/health")
print("GENERATE:", PUBLIC_URL + "/generate_multi")

public = requests.get(
    PUBLIC_URL + "/health",
    timeout=30,
)
public.raise_for_status()
print("PUBLIC HEALTH:", public.json())
